In [1]:
import grpc
import tensorflow as tf
from tensorflow_serving.apis import predict_pb2, prediction_service_pb2_grpc

2026-10-20 10:54:16.744909: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: .venv/lib/python3.12/site-packages/nvidia/cudnn/lib:.venv/lib/python3.12/site-packages/nvidia/cublas/lib
2026-10-20 10:54:16.744926: I tensorflow/stream_executor/cuda/cudart_stub.cc:29] Ignore above cudart dlerror if you do not have a GPU set up on your machine.


In [2]:
host = "localhost:8500"

channel = grpc.insecure_channel(host)

stub = prediction_service_pb2_grpc.PredictionServiceStub(channel)

In [4]:
from keras_image_helper import create_preprocessor

In [5]:
preprocessor = create_preprocessor("xception", target_size=(299, 299))

In [6]:
url = "http://bit.ly/mlbookcamp-pants"
X = preprocessor.from_url(url)

In [7]:
def np_to_protobuf(data):
    return tf.make_tensor_proto(data, shape=data.shape)

In [8]:
pb_request = predict_pb2.PredictRequest()

pb_request.model_spec.name = "clothing-model"
pb_request.model_spec.signature_name = "serving_default"

pb_request.inputs["input_8"].CopyFrom(np_to_protobuf(X))

In [9]:
pb_response = stub.Predict(pb_request, timeout=20.0)

In [10]:
preds = pb_response.outputs["dense_7"].float_val

In [11]:
classes = [
    "dress",
    "hat",
    "longsleeve",
    "outwear",
    "pants",
    "shirt",
    "shoes",
    "shorts",
    "skirt",
    "t-shirt",
]

In [12]:
dict(zip(classes, preds))

{'dress': -1.8798645734786987,
 'hat': -4.756311893463135,
 'longsleeve': -2.3595333099365234,
 'outwear': -1.0892642736434937,
 'pants': 9.90378475189209,
 'shirt': -2.826181173324585,
 'shoes': -3.648310422897339,
 'shorts': 3.2411563396453857,
 'skirt': -2.612095832824707,
 't-shirt': -4.852035999298096}